# 9주차 ③ 양자화 · 세 방식 비교 · Grad-CAM — 실습 6~8

**목표**: 4bit 양자화가 메모리를 줄이는 것을 **직접 측정**하고,
백본 동결·LoRA·전체 미세조정을 한 표로 비교해 **"나라면 무엇을 고를까"** 에 답하며,
Grad-CAM 으로 모델의 판단 근거를 보고 그 **한계**를 말한다.

```
   같은 가중치 0.3742 를
     fp32   4 byte   0.3742000...        지금까지
     fp16   2 byte   0.3742              7주차 AMP
     int8   1 byte   0.37 정도            ★ 1/4
     int4   0.5 byte 0.4 정도             ★ 1/8
        정밀도를 버리고 용량을 얻는 거래

   ViT-base 8,600만 개 기준
     fp32  →  약 344 MB      fp16  →  약 172 MB      4bit  →  약 43 MB   ★ 8배
```

```
   ┌──────────────────────────────────────────────┐
   │  백본 (8,600만)   ← 4bit 로 얼려 둔다         │  메모리 1/8
   │       +                                       │
   │  LoRA 어댑터      ← fp16 으로 학습한다  ★     │  작고 정밀하게
   └──────────────────────────────────────────────┘
              QLoRA = 4bit 백본 + fp16 어댑터
```

> **핵심 메시지 ★★ (기말 출제 지점)**: **"무엇을 양자화하고 무엇을 학습하는가"** 가 핵심 질문입니다.
> - **양자화하는 것** : 얼어 있는 백본 (어차피 갱신 안 하니 정밀도가 덜 중요)
> - **학습하는 것** : LoRA 어댑터 (갱신되므로 fp16 유지)
>
> **얼린 것은 대충, 배우는 것은 정밀하게.** 이 한 문장이 QLoRA 입니다.

| | 이 과목 (2학년) | 3학년 「최신인공지능」 |
|---|---|---|
| 양자화의 목적 | **학습**을 8GB 에 밀어 넣기 (QLoRA) | **추론**을 가볍게 (GGUF·Ollama) |

## 실습 6 — 4bit 로딩 + VRAM 측정

> ⚠️ **`bitsandbytes` 는 Windows 에서 실패할 수 있습니다.**
> 오류가 나면 **거기서 멈추고** 교수 제공 결과값을 비교표에 넣으세요.
> *"라이브러리가 OS·GPU 를 타는 것은 실무에서 늘 있는 일"* 입니다.
> **이 실습이 안 돼도 오늘의 목표는 달성됩니다.**

In [ ]:
# 셀 0 — 앞 교시에서 이어서 (커널을 재시작했다면 이 셀부터)
import torch, os, json, gc
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.transforms import v2
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

DATA = "mydata"
device = "cuda" if torch.cuda.is_available() else "cpu"
IMG = 224
MEAN, STD = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)

val_tf = v2.Compose([v2.Resize((IMG, IMG)),
                     v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])
train_set = ImageFolder(f"{DATA}/train", transform=val_tf)
val_set   = ImageFolder(f"{DATA}/val",   transform=val_tf)
val_loader = DataLoader(val_set, batch_size=16)
CLASSES = train_set.classes

baseline    = json.load(open("results/baseline.json", encoding="utf-8"))   # 1교시
lora_result = json.load(open("results/lora.json",     encoding="utf-8"))   # 2교시
print("클래스 :", CLASSES)
print("기준선 :", baseline["방식"], f"{baseline['정확도']*100:.2f}%")
print("LoRA   :", lora_result["방식"], f"{lora_result['정확도']*100:.2f}%")

In [ ]:
# 셀 1 — fp16 vs 4bit 메모리
from transformers import ViTForImageClassification, BitsAndBytesConfig

def load_and_measure(quant4bit):
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    kw = dict(num_labels=len(CLASSES), ignore_mismatched_sizes=True)
    if quant4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,      # ★ 계산은 fp16 으로
            bnb_4bit_quant_type="nf4",
        )
        kw["device_map"] = {"": 0}
    m = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224", **kw)
    if not quant4bit: m = m.to(device).half()
    used = torch.cuda.memory_allocated()/1024**3 if device == "cuda" else 0
    print(f"{'4bit' if quant4bit else 'fp16':6s} | 모델 로딩 후 VRAM {used:5.3f} GB")
    return m, used

m16, v16 = load_and_measure(False)
del m16; gc.collect()
if device == "cuda": torch.cuda.empty_cache()

m4,  v4  = load_and_measure(True)
print(f"\n→ 메모리 {v16/max(v4,1e-9):.2f}배 절감")

> **관찰 포인트 ★**: 4bit 쪽 VRAM 이 확실히 적습니다.
> 여기서 아낀 메모리로 **배치를 키우거나 더 큰 모델을 올릴 수 있습니다.**

> `bnb_4bit_compute_dtype=torch.float16` 에 주목하세요.
> **저장은 4bit, 계산은 fp16** 입니다. 실제 곱셈은 정밀도를 조금 올려서 합니다.

> 3주차에 *"GPU 는 빠르지만 좁다"* 고 했고, 7주차에 **AMP** 로 절반을 줄였습니다.
> 양자화는 **그다음 단계**입니다. *"메모리가 모자라면 fp16 → 그래도 모자라면 양자화"*.

## 실습 7 — 세 방식 비교표 완성 ★

**오늘의 결론이자 과제의 핵심 산출물.**

In [ ]:
# 셀 2 — 표로 정리
full_finetune = dict(          # ★ 교수 제공 (수업 중 실행하지 않음)
    방식="전체 미세조정",
    학습파라미터=86_000_000,
    정확도=0.00,               # ← 교수가 알려 주는 값으로 채우세요
    시간=0.0,
    VRAM=0.0,
)

rows = [baseline, lora_result, full_finetune]

print(f"{'방식':16s}{'학습 파라미터':>16s}{'비율':>8s}{'정확도':>9s}{'VRAM':>9s}{'시간':>9s}")
print("-" * 70)
for r in rows:
    print(f"{r['방식']:16s}{r['학습파라미터']:>16,d}"
          f"{r['학습파라미터']/86_000_000*100:>7.2f}%"
          f"{r['정확도']*100:>8.2f}%{r['VRAM']:>8.2f}G{r['시간']:>8.1f}s")

In [ ]:
# 셀 3 — results/comparison.md 로 저장
os.makedirs("results", exist_ok=True)
with open("results/comparison.md", "w", encoding="utf-8") as f:
    f.write("# 9주차 — 세 가지 파인튜닝 방식 비교\n\n")
    f.write(f"- 데이터 : {len(CLASSES)}클래스 / train {len(train_set)}장 / val {len(val_set)}장\n")
    f.write("- 모델 : google/vit-base-patch16-224\n")
    f.write(f"- 어댑터 {lora_result['어댑터MB']:.2f} MB vs 전체 모델 {lora_result['전체모델MB']:.1f} MB\n\n")
    f.write("| 방식 | 학습 파라미터 | 비율 | 정확도 | VRAM | 시간 |\n|---|---:|---:|---:|---:|---:|\n")
    for r in rows:
        f.write(f"| {r['방식']} | {r['학습파라미터']:,} "
                f"| {r['학습파라미터']/86_000_000*100:.2f}% "
                f"| {r['정확도']*100:.2f}% | {r['VRAM']:.2f} GB | {r['시간']:.1f}초 |\n")
    f.write("\n## 내 데이터라면 어느 방식을 쓰겠는가 (5줄)\n\n")
    f.write("> 여기에 직접 쓰세요 — 자원·데이터·목표 중 둘 이상을 근거로.\n")

print(open("results/comparison.md", encoding="utf-8").read())

> **관찰 포인트 ★★**: 표를 보고 **세 가지를 짚어 보세요.**
> ```
>   ① 백본 동결은 가장 싸지만 정확도가 아쉽다
>   ② LoRA 는 학습 파라미터가 1% 미만인데 정확도가 전체 미세조정에 근접한다  ★
>   ③ 전체 미세조정은 가장 좋지만 VRAM·시간이 압도적으로 크다
> ```

> **핵심 메시지 ★★**: **표를 만드는 것이 목적이 아닙니다.**
> *"내 상황이 이러하니 나는 이걸 고르겠다"* 는 **판단**이 목적입니다.
> ```
>   데이터 200장 + 8GB GPU + 오늘 안에 결과   →  ?
>   데이터 10만 장 + A100 + 최고 성능 필요     →  ?
> ```
> 과제의 마지막 5줄이 정확히 이 질문이고, **기말고사 D 구분(판단 근거)** 문항도 이것입니다.
> **답이 하나가 아닙니다.** 근거가 타당하면 됩니다.

## 실습 8 — Grad-CAM

```
   모델이 "고양이"라고 답했다.  →  그런데 무엇을 보고 그렇게 판단했나?

     Grad-CAM : 마지막 특징맵에 대한 기울기로 "중요한 영역"을 히트맵으로 그린다
                빨간 곳 = 판단에 크게 기여한 부분
```

> 7주차 실습 3에서 **특징맵**을 봤죠. 그건 *"층이 무엇에 반응하나"* 였습니다.
> Grad-CAM 은 한 발 더 나가 *"**이 판단**에 어디가 기여했나"* 를 봅니다.

In [ ]:
# 셀 4 — 2교시에 학습한 LoRA 모델을 되살린다
from peft import PeftModel

base = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224", num_labels=len(CLASSES), ignore_mismatched_sizes=True)
model = PeftModel.from_pretrained(base, "models/lora_adapter").to(device)
model.eval()
print("LoRA 모델 복원 완료")

In [ ]:
# 셀 5 — Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np

# ViT 는 마지막 블록의 LayerNorm 을 대상 층으로 잡는다
target_layers = [model.base_model.model.vit.encoder.layer[-1].layernorm_before]

def reshape_transform(tensor, h=14, w=14):          # ★ ViT 는 토큰 시퀀스라 2D 로 되돌린다
    r = tensor[:, 1:, :].reshape(tensor.size(0), h, w, tensor.size(2))
    return r.permute(0, 3, 1, 2)

cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=reshape_transform)

xb, yb = next(iter(val_loader))
fig, ax = plt.subplots(2, 3, figsize=(10, 7))
for i in range(3):
    x1 = xb[i:i+1].to(device)
    grayscale = cam(input_tensor=x1)[0]
    rgb = (xb[i].permute(1,2,0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0,1).numpy()
    with torch.no_grad():
        pred = model(x1).logits.argmax(1).item()

    ax[0, i].imshow(rgb); ax[0, i].axis("off")
    ax[0, i].set_title(f"정답 {CLASSES[yb[i]]}")
    ax[1, i].imshow(show_cam_on_image(rgb.astype(np.float32), grayscale, use_rgb=True))
    ax[1, i].axis("off"); ax[1, i].set_title(f"예측 {CLASSES[pred]}")
plt.tight_layout(); plt.show()

In [ ]:
# 셀 6 — 저장 (과제 제출물)
os.makedirs("outputs", exist_ok=True)
fig.savefig("outputs/gradcam.png", dpi=120, bbox_inches="tight")
print("저장 완료 : outputs/gradcam.png")

> **관찰 포인트 ★**: 히트맵이 **물체 위에 있으면** 모델이 제대로 보고 있는 것입니다.
> **배경에 있으면 위험 신호**입니다 — 예를 들어 강아지 사진이 전부 잔디밭이면
> 모델이 *"잔디 = 강아지"* 를 배웠을 수 있습니다. **데이터 편향**입니다.

> **핵심 메시지 ★ (출제 지점)**: **Grad-CAM 의 한계** —
> *"어디를 봤다"* 는 알려 주지만 ***"왜 그렇게 판단했는지"*** 는 알려 주지 않습니다.
> 히트맵이 그럴듯해 보여도 모델이 옳은 근거로 판단했다는 보장은 없습니다.
> **설명 가능성 도구는 참고 자료이지 증명이 아닙니다.**

> ⚠️ 대상 층·`reshape_transform` 은 모델 구조를 탑니다. 오류가 나면
> `print(model)` 로 층 경로를 확인하세요. 실패해도 이 실습의 목적은 코딩이 아니라 **해석**입니다.

---

### 과제 (마감 11/5 목 23:59) — **과제 15점 중 1.5점, 주차별 최고 배점**

```
  ① 15_lora_finetune.ipynb (출력 저장)
  ② 비교표 — 백본 동결 / LoRA / 전체 미세조정
       × 학습 파라미터 수 · VRAM · 정확도 · 시간
  ③ 어댑터 폴더 용량 vs 전체 모델 용량 대비          ★
  ④ Grad-CAM 결과 3장
  ⑤ "내 데이터라면 어느 방식을 쓰겠는가" 근거 5줄    ★★ 이게 핵심
  ⑥ 커밋 · push · LMS 제출
```

> **과제의 핵심은 표가 아니라 ⑤의 5줄입니다.**
> 정답은 없습니다. **근거가 자원·데이터·목표 중 둘 이상을 짚으면 만점**입니다.

> ⚠️ `mydata/` 와 모델 가중치는 `.gitignore` 입니다.
> 단 **어댑터는 수 MB 라 커밋을 권장**합니다.

### 이 노트북 체크리스트

- [ ] 양자화가 메모리를 줄이는 원리를 말할 수 있다
- [ ] **QLoRA 에서 무엇을 양자화하고 무엇을 학습하는지** 안다 ★★
- [ ] 4bit 로딩 VRAM 을 측정했다 (또는 교수 제공값을 받았다)
- [ ] **세 방식 비교표를 완성**하고 `results/comparison.md` 로 저장했다 ★
- [ ] "내 상황이면 무엇을 고를지" 근거를 댈 수 있다 ★★
- [ ] Grad-CAM 히트맵을 그리고 해석했다
- [ ] Grad-CAM 의 한계를 말할 수 있다
- [ ] 학습용 양자화와 추론용 양자화의 차이를 안다